# fractional-stride-zero-insertion — faded example 1: 2-D Zero Insertion: Allocate and Scatter

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `fractional-stride-zero-insertion`. Running the beacon reports progress on the `CNN: ConvT fractional-stride zero insertion` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ConvT fractional-stride zero insertion` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`fractional-stride-zero-insertion`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "fractional-stride-zero-insertion"
DD_SUBTOPIC = "CNN: ConvT fractional-stride zero insertion"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The core of ConvTranspose2d's zero-insertion step is two lines: (1) allocate a zeros tensor with the dilated output shape `(B, C, (H-1)*s+1, (W-1)*s+1)`, and (2) scatter the input into every `s`-th position along both spatial axes with `y[:, :, ::s, ::s] = x`. Every other position stays zero.

## Faded exercise 1

Implement `zero_insert(x, s)` for `x: (B, C, H, W)`. Compute the output shape, allocate a zeros tensor of that shape with `x`'s dtype, scatter the input with step-`s` slicing, and return the result.

**Fill in:** Allocate y as t.zeros of shape (B, C, H_out, W_out) with x.dtype, then assign x into y using the stride-s slice y[:, :, ::s, ::s] = x.

In [ ]:
import torch as t
from torch import Tensor

def zero_insert(x: Tensor, s: int) -> Tensor:
    B, C, H, W = x.shape
    H_out = (H - 1) * s + 1
    W_out = (W - 1) * s + 1
    y = t.zeros(B, C, H_out, W_out, dtype=x.dtype)
    y[:, :, ::s, ::s] = x
    return y


def _test():
    import torch as t
    import torch.nn.functional as F
    t.manual_seed(3)
    x = t.randn(1, 1, 4, 4)
    y = zero_insert(x, s=2)
    assert y.shape == (1, 1, 7, 7)
    # original pixels at ::2 positions
    assert t.allclose(y[:, :, ::2, ::2], x)
    # zeros elsewhere
    assert y[:, :, 1, :].abs().sum().item() == 0
    # equivalence check
    w = t.randn(1, 1, 3, 3)
    out_a = F.conv_transpose2d(y, w, stride=1)
    out_b = F.conv_transpose2d(x, w, stride=2)
    assert t.allclose(out_a, out_b, atol=1e-5)


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
from torch import Tensor

def zero_insert(x: Tensor, s: int) -> Tensor:
    B, C, H, W = x.shape
    H_out = (H - 1) * s + 1
    W_out = (W - 1) * s + 1
    y = t.zeros(B, C, H_out, W_out, dtype=x.dtype)
    y[:, :, ::s, ::s] = x
    return y
```
</details>